# Scene 2 Full-Frame Experimental Gemini Realization — front-context + behind-black masking

This notebook keeps the original full-frame pipeline and applies the mask/control logic we discussed:

- build depth fallback pixels for each layer
- define each layer's **improved support** as `visible_mask OR depth_fallback_mask`
- when generating the current layer, **show**:
  - the current layer improved support
  - the improved support of layers in front
- **black out**:
  - the visible parts of layers behind
- leave everything else beige

In [ ]:
%cd /content
!git clone https://github.com/UrbinaDan/PaperTheater_ai_Pipeline.git
%cd PaperTheater_ai_Pipeline

!pip install -q numpy scipy matplotlib opencv-python-headless pillow shapely svgwrite cairosvg tqdm networkx pyyaml requests google-genai openai accelerate bitsandbytes einops sentencepiece transformers==4.49.0

import os
from getpass import getpass

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

if "OPENAI_API_KEY" not in os.environ:
    print("OPENAI_API_KEY not set. OpenAI layer-plan generation will be unavailable unless you set it.")

%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -e .
!mkdir -p checkpoints
!wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
!wget -q -P checkpoints https://raw.githubusercontent.com/facebookresearch/sam2/main/sam2_configs/sam2_hiera_s.yaml

%cd /content
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
%cd Depth-Anything-V2
!mkdir -p checkpoints
!wget -q -P checkpoints https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth

%cd /content/PaperTheater_ai_Pipeline

In [ ]:
import os
import io
import json
import time
import base64
import importlib
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from google import genai
from google.genai.errors import ServerError, ClientError
from openai import OpenAI

from src.config import Paths, PipelineConfig
from src.io_utils import ensure_dirs, load_image, save_mask
from src.florence_parser import FlorenceParser
from src.sam2_segmenter import SAM2Segmenter
from src.depth_anything_runner import DepthRunner
from src.occlusion_heuristic import heuristic_complete
from src.mask_cleanup import cleanup_mask

import src.scene_builder
importlib.reload(src.scene_builder)
from src.scene_builder import (
    parse_florence_boxes,
    merge_segmented_by_label,
    build_stable_merged_objects,
)

import src.scene_representation
importlib.reload(src.scene_representation)
from src.scene_representation import build_scene_representation, save_scene_representation

import src.layer_planner
importlib.reload(src.layer_planner)
from src.layer_planner import plan_layers_deterministic

import src.layer_renderer
importlib.reload(src.layer_renderer)
from src.layer_renderer import build_object_mask_map

import src.layer_context_builder
importlib.reload(src.layer_context_builder)
from src.layer_context_builder import build_layer_contexts

paths = Paths()
cfg = PipelineConfig()
ensure_dirs(paths)

gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) if os.environ.get("OPENAI_API_KEY") else None

In [ ]:
SCENE_NAME = "scene_2"
image_path = f"/content/PaperTheater_ai_Pipeline/data/input/{SCENE_NAME}.jpg"

SCENE_REPR_PATH = f"/content/PaperTheater_ai_Pipeline/data/intermediate/{SCENE_NAME}_scene_repr.json"
LAYER_PLAN_PATH = f"/content/PaperTheater_ai_Pipeline/data/intermediate/{SCENE_NAME}_layer_plan.json"
FULLFRAME_OUTPUT_DIR = f"/content/PaperTheater_ai_Pipeline/data/intermediate/{SCENE_NAME}_fullframe_frontcontext_behindblack_gemini"
LAYER_PLAN_DIR = f"/content/PaperTheater_ai_Pipeline/data/intermediate/{SCENE_NAME}_llm_layer_plans"

GEMINI_IMAGE_MODEL = "gemini-3.1-flash-image-preview"
LAYER_PLAN_PROVIDER = "gemini"
OPENAI_TEXT_MODEL = "gpt-5"
GEMINI_TEXT_MODEL = "gemini-2.5-flash"
USE_LLM_LAYER_PLAN = True

STYLE_NORMALIZATION_PROMPT = """
Render this as a stylized Japanese paper theater / kamishibai layer.

GLOBAL STYLE RULES:
- flat paper-cut illustration
- simplified silhouettes and clean edges
- muted earthy palette
- minimal texture
- no photorealism
- no realistic skin or fur rendering
- no detailed lighting or harsh shadows
- consistent abstraction across all layers
- all layers should look like they were made by the same artist
- preserve the original composition and scale inside the full canvas
- output should feel like a clean cuttable layer for paper theater production
""".strip()

In [ ]:
def expand_mask(mask, kernel_size=61):
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    expanded = cv2.dilate(mask.astype(np.uint8), kernel, iterations=1)
    return expanded > 0

def normalize_to_rgb_array(img_obj):
    if isinstance(img_obj, np.ndarray):
        arr = img_obj
        if arr.ndim == 2:
            return np.stack([arr] * 3, axis=-1).astype(np.uint8)
        if arr.shape[-1] == 4:
            return arr[..., :3].astype(np.uint8)
        return arr.astype(np.uint8)
    if isinstance(img_obj, Image.Image):
        return np.array(img_obj.convert("RGB"))
    if isinstance(img_obj, (bytes, bytearray)):
        return np.array(Image.open(io.BytesIO(img_obj)).convert("RGB"))
    if hasattr(img_obj, "image_bytes") and img_obj.image_bytes is not None:
        return np.array(Image.open(io.BytesIO(img_obj.image_bytes)).convert("RGB"))
    if hasattr(img_obj, "numpy_view"):
        arr = np.array(img_obj.numpy_view())
        if arr.ndim == 2:
            return np.stack([arr] * 3, axis=-1).astype(np.uint8)
        if arr.shape[-1] == 4:
            return arr[..., :3].astype(np.uint8)
        return arr.astype(np.uint8)
    raise TypeError(f"Unsupported image object type: {type(img_obj)}")

def save_bool_mask(mask, path):
    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)

def mask_to_rgba_fullframe(image_rgb, mask):
    alpha = (mask.astype(np.uint8)) * 255
    return np.dstack([image_rgb, alpha])

def compute_layer_depth_stats(layer_contexts, depth_map):
    stats = {}
    for ctx in layer_contexts:
        ownership_mask = ctx["ownership_mask"] > 0
        vals = depth_map[ownership_mask]
        rep_depth = float(np.median(vals)) if vals.size else float(np.median(depth_map))
        stats[ctx["layer_name"]] = {"order": ctx["order"], "rep_depth": rep_depth}
    return stats

def build_unassigned_pixel_mask(layer_contexts, image_shape):
    H, W = image_shape[:2]
    assigned = np.zeros((H, W), dtype=bool)
    for ctx in layer_contexts:
        assigned |= (ctx["ownership_mask"] > 0)
    return ~assigned

def assign_leftover_pixels_by_depth(layer_contexts, depth_map):
    H, W = depth_map.shape
    layer_stats = compute_layer_depth_stats(layer_contexts, depth_map)
    unassigned = build_unassigned_pixel_mask(layer_contexts, (H, W))
    fallback_masks = {ctx["layer_name"]: np.zeros((H, W), dtype=bool) for ctx in layer_contexts}
    layer_names = [ctx["layer_name"] for ctx in layer_contexts]
    layer_depths = np.array([layer_stats[name]["rep_depth"] for name in layer_names], dtype=np.float32)

    ys, xs = np.where(unassigned)
    if len(ys) == 0:
        return fallback_masks, unassigned, layer_stats

    pixel_depths = depth_map[ys, xs].astype(np.float32)
    dist = np.abs(pixel_depths[:, None] - layer_depths[None, :])
    best_idx = np.argmin(dist, axis=1)

    for k, layer_idx in enumerate(best_idx):
        fallback_masks[layer_names[layer_idx]][ys[k], xs[k]] = True

    return fallback_masks, unassigned, layer_stats

def restrict_fallback_masks(layer_contexts, depth_fallback_masks):
    restricted = {}
    small_object_labels = {"person", "deer", "building"}
    for ctx in layer_contexts:
        layer_name = ctx["layer_name"]
        ownership_mask = ctx["ownership_mask"] > 0
        fallback_mask = depth_fallback_masks[layer_name] > 0
        labels = set(ctx.get("labels", []))
        if labels & small_object_labels:
            local_window = expand_mask(ownership_mask.astype(np.uint8), kernel_size=151)
            fallback_mask = fallback_mask & local_window
        restricted[layer_name] = fallback_mask
    return restricted

def get_layer_support_mask(layer_context, depth_fallback_masks):
    visible_mask = layer_context["visible_mask"] > 0
    fallback_mask = depth_fallback_masks[layer_context["layer_name"]] > 0
    return visible_mask | fallback_mask

def build_control_masks_for_layer(layer_context, ordered_contexts, depth_fallback_masks):
    order = layer_context["order"]
    layer_name = layer_context["layer_name"]

    visible_mask = layer_context["visible_mask"] > 0
    fallback_mask = depth_fallback_masks[layer_name] > 0
    current_support = visible_mask | fallback_mask

    H, W = visible_mask.shape
    front_support = np.zeros((H, W), dtype=bool)
    behind_visible = np.zeros((H, W), dtype=bool)

    for ctx in ordered_contexts:
        if ctx["layer_name"] == layer_name:
            continue
        if ctx["order"] < order:
            front_support |= get_layer_support_mask(ctx, depth_fallback_masks)
        elif ctx["order"] > order:
            behind_visible |= (ctx["visible_mask"] > 0)

    show_mask = current_support | front_support
    black_mask = behind_visible.copy()
    beige_mask = ~(show_mask | black_mask)

    return {
        "anchor_mask": visible_mask,
        "fallback_mask": fallback_mask,
        "current_support": current_support,
        "front_support": front_support,
        "behind_visible": behind_visible,
        "show_mask": show_mask,
        "black_mask": black_mask,
        "beige_mask": beige_mask,
    }

def apply_focus_mask_fullframe(image_rgb, show_mask, black_mask, beige_value=235):
    h, w = show_mask.shape
    out = np.full((h, w, 3), beige_value, dtype=np.uint8)
    out[black_mask > 0] = 0
    out[show_mask > 0] = image_rgb[show_mask > 0]
    return out

def show_mask_debug(image_rgb, layer_name, anchor_mask, fallback_mask, current_support, front_support, behind_visible, show_mask, black_mask, focused_input):
    h, w = anchor_mask.shape
    comp = np.zeros((h, w, 3), dtype=np.uint8)
    comp[anchor_mask] = [80, 140, 255]
    comp[fallback_mask] = [220, 80, 220]
    comp[front_support] = [80, 220, 220]
    comp[behind_visible] = [220, 80, 80]

    fig, axes = plt.subplots(2, 3, figsize=(16, 18))
    axes = axes.ravel()
    axes[0].imshow(image_rgb); axes[0].set_title(f"{layer_name} - original"); axes[0].axis("off")
    axes[1].imshow(current_support, cmap="gray"); axes[1].set_title("current_support"); axes[1].axis("off")
    axes[2].imshow(front_support, cmap="gray"); axes[2].set_title("front_support"); axes[2].axis("off")
    axes[3].imshow(behind_visible, cmap="gray"); axes[3].set_title("behind_visible (black)"); axes[3].axis("off")
    axes[4].imshow(comp); axes[4].set_title("blue=anchor magenta=fallback cyan=front support red=behind blacked"); axes[4].axis("off")
    axes[5].imshow(focused_input); axes[5].set_title("focused_input sent to Gemini"); axes[5].axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
image = load_image(image_path, max_side=cfg.image_max_side)

plt.figure(figsize=(8, 12))
plt.imshow(image)
plt.axis("off")
plt.title(SCENE_NAME)
plt.show()

print("image shape:", image.shape)
print("output dir:", FULLFRAME_OUTPUT_DIR)
print("gemini image model:", GEMINI_IMAGE_MODEL)
print("layer-plan provider:", LAYER_PLAN_PROVIDER)

florence = FlorenceParser(cfg.florence_model)
segmenter = SAM2Segmenter(cfg.sam2_config, cfg.sam2_checkpoint)
depth_runner = DepthRunner(cfg.depth_encoder)

In [ ]:
caption = florence.get_dense_caption(image)
print(caption)

queries = ["a person", "a deer", "trees", "a building", "sky"]

all_boxes = []
for q in queries:
    det_q = florence.get_open_vocab_detection(image, q)
    boxes_q = parse_florence_boxes(det_q, image.shape)
    print("\nQUERY:", q)
    print("RAW:", det_q)
    print("PARSED:", boxes_q)
    all_boxes.extend(boxes_q)

seen = set()
boxes = []
for b in all_boxes:
    key = (b["label"], tuple(b["bbox"]))
    if key not in seen:
        seen.add(key)
        boxes.append(b)

print("\nFINAL BOXES:")
print(boxes)
print("num boxes:", len(boxes))

fig, ax = plt.subplots(figsize=(8, 12))
ax.imshow(image)
for b in boxes:
    x1, y1, x2, y2 = b["bbox"]
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, b["label"], color="yellow", fontsize=10, backgroundcolor="black")
ax.axis("off")
plt.show()

segmented = segmenter.segment_boxes(image, boxes)
merged_segmented = merge_segmented_by_label(segmented)

depth = depth_runner.infer(image)
stable_objects = build_stable_merged_objects(merged_segmented, depth)

In [ ]:
scene_repr = build_scene_representation(
    image_path=image_path,
    image_shape=image.shape,
    caption=caption,
    stable_objects=stable_objects,
)

save_scene_representation(scene_repr, SCENE_REPR_PATH)
layer_plan = plan_layers_deterministic(scene_repr)

with open(LAYER_PLAN_PATH, "w", encoding="utf-8") as f:
    json.dump(layer_plan, f, indent=2)

mask_records = []
for obj in stable_objects:
    cleaned_mask = cleanup_mask(
        heuristic_complete(obj["mask"], obj["label"]),
        cfg.min_component_area,
        cfg.smooth_kernel
    )
    out_mask = paths.completed_dir / f"{SCENE_NAME}_mask_{obj['id']}.png"
    save_mask(out_mask, cleaned_mask)
    mask_records.append({
        "id": obj["id"],
        "label": obj["label"],
        "bbox": obj["bbox"],
        "mask_path": str(out_mask)
    })

object_mask_map = build_object_mask_map(mask_records)

layer_contexts = build_layer_contexts(
    scene_repr=scene_repr,
    layer_plan=layer_plan,
    object_mask_map=object_mask_map,
)

depth_fallback_masks, unassigned_pixels_mask, layer_depth_stats = assign_leftover_pixels_by_depth(layer_contexts, depth)
depth_fallback_masks = restrict_fallback_masks(layer_contexts, depth_fallback_masks)

In [ ]:
def build_layer_plan_request(layer_context, scene_repr):
    labels = layer_context.get("labels", [])
    primary_label = labels[0] if labels else layer_context["layer_name"]
    return {
        "scene_caption": layer_context.get("scene_caption", scene_repr.get("caption", "")),
        "image_shape": scene_repr.get("image_shape"),
        "layer_name": layer_context["layer_name"],
        "primary_label": primary_label,
        "labels": labels,
        "object_ids": layer_context.get("object_ids", []),
        "front_layer_names": layer_context.get("front_layer_names", []),
        "ownership_bbox": layer_context.get("ownership_bbox"),
        "visible_bbox": layer_context.get("visible_bbox"),
    }

def build_layer_plan_prompt(layer_context, scene_repr):
    payload = build_layer_plan_request(layer_context, scene_repr)
    return f"""
You are planning one image layer for a paper-theater generation pipeline.

Return STRICT JSON only with this exact schema:
{{
  "layer_goal": "short phrase",
  "must_include": ["item1", "item2"],
  "allowed_extensions": ["item1", "item2"],
  "forbidden_elements": ["item1", "item2"],
  "count_guidance": "short sentence",
  "density_guidance": "short sentence",
  "placement_guidance": "short sentence",
  "style_guidance": "short sentence"
}}

Layer context JSON:
{json.dumps(payload, ensure_ascii=False, indent=2)}
""".strip()

In [ ]:
def layer_plan_to_text(layer_context):
    return ""

def build_fullframe_prompt(layer_context):
    label = layer_context["labels"][0] if layer_context["labels"] else layer_context["layer_name"]
    return f"""
Generate a FULL-FRAME layer for a layered paper-theater scene.

Scene context:
{layer_context["scene_caption"]}

Target layer:
- layer name: {layer_context["layer_name"]}
- semantic label: {label}

{STYLE_NORMALIZATION_PROMPT}

INPUT INTERPRETATION:
- The current target layer and the improved support of front layers are shown with original image pixels.
- Black regions correspond to visible areas from layers behind the current layer.
- Beige regions are neutral canvas where you may extend and complete the current target layer if helpful.
- Do not copy front occluders into the target layer output.

IMPORTANT RULES:
1. Generate the target layer only.
2. Preserve exact scene alignment with the original image.
3. Do not invent unrelated objects from other layers.
4. Use front-layer context only as context.
5. Black regions correspond to behind-layer visible regions and should remain excluded from the target layer.
""".strip()

In [ ]:
def gemini_edit(image_rgb, prompt, model_name, max_retries=4):
    pil_img = Image.fromarray(image_rgb.astype(np.uint8))
    last_err = None
    for attempt in range(max_retries):
        try:
            response = gemini_client.models.generate_content(model=model_name, contents=[prompt, pil_img])

            if hasattr(response, "generated_images") and response.generated_images:
                img_obj = response.generated_images[0].image
                if hasattr(img_obj, "image_bytes") and img_obj.image_bytes is not None:
                    raw = img_obj.image_bytes
                    if isinstance(raw, str):
                        raw = base64.b64decode(raw)
                    return np.array(Image.open(io.BytesIO(raw)).convert("RGB"))

            raise ValueError("Gemini response did not contain a usable image")
        except ServerError as e:
            last_err = e
            time.sleep(min(60, 5 * (2 ** attempt)))
        except ClientError:
            raise
    raise last_err

In [ ]:
def realize_single_layer_fullframe(image, layer_context, output_dir, model_name):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    ordered_contexts = sorted(layer_contexts, key=lambda x: x["order"])
    order = layer_context["order"]
    layer_name = layer_context["layer_name"]

    mask_pack = build_control_masks_for_layer(layer_context, ordered_contexts, depth_fallback_masks)

    anchor_mask = mask_pack["anchor_mask"]
    fallback_mask = mask_pack["fallback_mask"]
    current_support = mask_pack["current_support"]
    front_support = mask_pack["front_support"]
    behind_visible = mask_pack["behind_visible"]
    show_mask = mask_pack["show_mask"]
    black_mask = mask_pack["black_mask"]

    visible_mask = layer_context["visible_mask"] > 0

    focused_input = apply_focus_mask_fullframe(
        image_rgb=image,
        show_mask=show_mask,
        black_mask=black_mask,
        beige_value=235,
    )

    prompt = build_fullframe_prompt(layer_context)
    generated_full = gemini_edit(focused_input, prompt=prompt, model_name=model_name)

    target_h, target_w = focused_input.shape[:2]
    if generated_full.shape[:2] != (target_h, target_w):
        generated_full = np.array(
            Image.fromarray(generated_full.astype(np.uint8)).resize(
                (target_w, target_h),
                Image.Resampling.LANCZOS
            )
        )

    generated_full_path = output_dir / f"{order:02d}_{layer_name}_generated_full.png"
    generated_visible_full_path = output_dir / f"{order:02d}_{layer_name}_generated_visible_full.png"
    Image.fromarray(generated_full).save(generated_full_path)
    Image.fromarray(mask_to_rgba_fullframe(generated_full, visible_mask)).save(generated_visible_full_path)

    return {
        "layer_name": layer_name,
        "order": order,
        "generated_full_path": str(generated_full_path),
        "generated_visible_full_path": str(generated_visible_full_path),
    }

In [ ]:
ordered_contexts = sorted(layer_contexts, key=lambda x: x["order"])

for ctx in ordered_contexts:
    layer_name = ctx["layer_name"]

    mask_pack = build_control_masks_for_layer(ctx, ordered_contexts, depth_fallback_masks)

    focused_input = apply_focus_mask_fullframe(
        image_rgb=image,
        show_mask=mask_pack["show_mask"],
        black_mask=mask_pack["black_mask"],
        beige_value=235,
    )

    print("=" * 60)
    print("LAYER:", layer_name)
    print("order:", ctx["order"])
    print("anchor sum:", int(mask_pack["anchor_mask"].sum()))
    print("fallback sum:", int(mask_pack["fallback_mask"].sum()))
    print("current_support sum:", int(mask_pack["current_support"].sum()))
    print("front_support sum:", int(mask_pack["front_support"].sum()))
    print("behind_visible sum:", int(mask_pack["behind_visible"].sum()))
    print("show_mask sum:", int(mask_pack["show_mask"].sum()))
    print("black_mask sum:", int(mask_pack["black_mask"].sum()))

    show_mask_debug(
        image_rgb=image,
        layer_name=layer_name,
        anchor_mask=mask_pack["anchor_mask"],
        fallback_mask=mask_pack["fallback_mask"],
        current_support=mask_pack["current_support"],
        front_support=mask_pack["front_support"],
        behind_visible=mask_pack["behind_visible"],
        show_mask=mask_pack["show_mask"],
        black_mask=mask_pack["black_mask"],
        focused_input=focused_input,
    )

In [ ]:
fullframe_results = []
failed_layers = []

for ctx in layer_contexts:
    try:
        result = realize_single_layer_fullframe(
            image=image,
            layer_context=ctx,
            output_dir=FULLFRAME_OUTPUT_DIR,
            model_name=GEMINI_IMAGE_MODEL,
        )
        fullframe_results.append(result)
    except Exception as e:
        failed_layers.append({"layer_name": ctx["layer_name"], "error": str(e)})

print("num success:", len(fullframe_results))
print("num failed:", len(failed_layers))
print("failed_layers:", failed_layers)

In [ ]:
for result in fullframe_results:
    print(result["layer_name"])
    for name, p in [
        ("generated_full", result["generated_full_path"]),
        ("generated_visible_full", result["generated_visible_full_path"]),
    ]:
        img = Image.open(p)
        plt.figure(figsize=(8, 10))
        plt.imshow(img)
        plt.title(f'{result["layer_name"]} — {name}')
        plt.axis("off")
        plt.show()